In [3]:
pip install openai langchain langchain-openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
import os

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0.1
)

print("✓ Modelo configurado con streaming habilitado")
print(f"Modelo: {llm.model_name}")
print(f"Streaming: {llm.streaming}")

✓ Modelo configurado con streaming habilitado
Modelo: gpt-4o
Streaming: True


In [3]:
history_store = {}

def get_session_history(session_id):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto en acuicultura para Salmones Camanchaca.
    Tu objetivo es apoyar al equipo con análisis de producción, salud de peces y logística
    en los centros de cultivo de la región de Los Lagos. Responde de forma técnica y precisa."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

In [4]:
session_id = "camanchaca_sesion_01"

print("1. Primera interacción:")
response1 = conversation.invoke(
    {"input": "Mi nombre es Carlos y soy jefe de producción del centro Ensenada. Trabajamos con Salmón Atlántico."},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nRespuesta: {response1.content}\n")

print("2. Segunda interacción:")
response2 = conversation.invoke(
    {"input": "¿Cuál es mi nombre y en qué centro trabajo?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nRespuesta: {response2.content}\n")

print("3. Tercera interacción:")
response3 = conversation.invoke(
    {"input": "Hoy cosechamos 5000 unidades con peso promedio de 4.5kg y 2% de mortalidad. ¿Cómo evalúas estos datos?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nRespuesta: {response3.content}\n")

print("CONTENIDO DE LA MEMORIA:\n")
for msg in history_store[session_id].messages:
    print(msg)

1. Primera interacción:
Encantado de apoyarte, Carlos. Si estás trabajando con Salmón Atlántico en el centro Ensenada, puedo ayudarte con análisis de producción, estrategias de manejo sanitario, optimización de la logística o cualquier otro aspecto técnico que necesites. Por favor, indícame en qué área específica necesitas asistencia o si tienes algún desafío particular en el centro.
Respuesta: Encantado de apoyarte, Carlos. Si estás trabajando con Salmón Atlántico en el centro Ensenada, puedo ayudarte con análisis de producción, estrategias de manejo sanitario, optimización de la logística o cualquier otro aspecto técnico que necesites. Por favor, indícame en qué área específica necesitas asistencia o si tienes algún desafío particular en el centro.

2. Segunda interacción:
Tu nombre es Carlos y trabajas como jefe de producción en el centro Ensenada, donde cultivan Salmón Atlántico. ¿En qué puedo asistirte específicamente?
Respuesta: Tu nombre es Carlos y trabajas como jefe de producc